### This script performs extensive and thorough data extraction, data cleaning, data formatting steps on raw data in OHCLV form collected from YFinance API.
1. Data Extraction to extract real-world historical stock market data on the 30 constituents in Dow Jones Industrial Average.
2. Data Formatting to transform raw, messy, unfit data frame based on API data extraction into meaningful and useful structured data format for further data engineering.

Details:

1. Adding day-of-week column to address the "day effects" in financial market:
      
      a. Monday Effect:
          Stocks often perform poorly on Mondays
          "Weekend news" impact

       b.  Friday Effect:
            Different trading patterns before weekends
            Position closing before weekends


      c.  End-of-week patterns:
        Higher volatility on certain days
        Institutional trading patterns


### Extract 29 constituents in Dow Jones Industrial Average, on HOLCV values from yfinance:

3M (MMM), American Express (AXP), Amgen (AMGN), Apple (AAPL), Boeing (BA), Caterpillar (CAT), Chevron (CVX), Cisco (CSCO), Coca-Cola (KO), Disney (DIS), Goldman Sachs (GS), Home Depot (HD), Honeywell (HON), IBM (IBM), Johnson & Johnson (JNJ), JPMorgan Chase (JPM), McDonald’s (MCD), Merck (MRK), Microsoft (MSFT), Nike (NKE), Procter & Gamble (PG), Salesforce (CRM), Sherwin-Williams (SHW) (added Nov 8 2024), Travelers (TRV), UnitedHealth (UNH), Verizon (VZ), Visa (V), and Walmart (WMT), WBA(Walgreens Boots Alliance) (dropped on Feburary 2024 when AMZN was added)

Note: This list reflects the constituents piror to Feburary 2024.
WBA is present and AMZN, NVDA are missing from the constituents based on August, 2025.

# install required packages
!pip install swig
!pip install wrds
!pip install pyportfolioopt
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

# !python3 -m venv ~/yfinance
# !source ~/yfinance/bin/activate
# !pip3 install yfinance

In [1]:
import pandas as pd
import numpy as np
import datetime
import yfinance as yf

import itertools

### Step1: Data Extraction

Using yfinance, an open source library that provides APIs fetching historical data from Yahoo Finance. 

Raw data is extracted in OHLCV format, corresponding to open, high, low, close, volume. 
OHLCV contains most of numerical information of the stock in time series.
From OHLCV, traders can get further judgement and predictions calculated through technical indicators, turbulence, vix.

NOTE:
1. the FinRL library's config_tickers.DOW_30_TICKER produces 
['AXP', 'AMGN', 'AAPL', 'BA', 'CAT', 'CSCO', 'CVX', 'GS', 'HD', 'HON', 'IBM', 'INTC', 'JNJ', 'KO', 'JPM', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'PG', 'TRV', 'UNH', 'CRM', 'VZ', 'V', 'WBA', 'WMT', 'DIS', 'DOW'].
This list is an older composition and doesn't reflect current constituents. Therefore, this list is edited manually to reflect the most updated version of the 30 constituents in Dow Jone, as of June 2025.
3. 

In [58]:
from pathlib import Path
import os

start_date = '2000-01-01'
end_date = '2025-08-01'

file_dir = (Path.cwd().parent/'datasets')
file_dir.mkdir(exist_ok=True)
train_file_path = file_dir / 'train.parquet'
test_file_path = file_dir / 'test.parquet'

dow_30_stocks = ['MMM', 'AXP', 'AMGN', 'AMZN', 'AAPL', 'BA', 'CAT', 'CVX', 'CSCO', 'KO', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'MCD', 'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'CRM', 'SHW', 'TRV', 'UNH', 'VZ', 'V', 'WMT']

In [3]:
# market_indicator = ['^VIX']
# data_list = dow_30_stocks + market_indicator

# print(data_list)

In [4]:
def extract_yf_rawdata(start_date, end_date, stock_list):
    """
    Follows defensive programming practice to deliver more robust data pipeline, during external API extraction.
    This function is modified for integration and is based on https://github.com/AI4Finance-Foundation/FinRL/blob/master/finrl/meta/preprocessor/yahoodownloader.py
    The result returns a data set with Datetime as index value, and an unintended 'Price' col formed due to MultiIndex metadata during data extraction process in Yfinance.

    Extracted:
    OHLCV prices, Date, VIX (Volatility Index aka market sentiment) real-time market index of expectations of 30-day forward volatility derived from S&P500 options prices.

    VIX would be a powerful macro-economic feature that captures market-wide risk sentiment, unavilable in individual stock data.
        e.g. During market crashes (2008, COVID-19) VIX spikes to 70-80+, while individual stock correlations approach 1.0.
        This makes VIX a critical regime-detection signal. High VIX periods often coincide with oversold bounces and volatility clustering effects.
        High VIX periods impact all equity returns.
        This provides a valuable feature for DRL Agent to learn market sentiment and make better decisions.
    
    Integration based on: https://github.com/AI4Finance-Foundation/FinRL/blob/master/finrl/meta/preprocessor/yahoodownloader.py#L36
    """
    data = pd.DataFrame()
    num_failures=0 
    
    for stock in stock_list:
        df = yf.download( #https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html#yfinance.download
            stock,
            start=start_date,
            end=end_date,
            auto_adjust=True,
        )
        # print(f"Stock {stock} DF: \n{df}\n")
        # print(f"Stock {stock} DF Columns: \n{df.columns}")
        # print(f"df columns: {df.columns.tolist()}")
        
        if df.columns.nlevels != 1: # df from the download would contain 2 levels, e.g.: ('Close', current ticker name)
            df.columns = df.columns.droplevel(1) #most memory and time efficient way, optimized to reduce columns
            df.columns.name=None #This removes the 'Price' MultiIndex metadata, which is empty
        df['Stock']=stock
        
        if not df.empty: #prevents empty df added to the final dataset
            data=pd.concat([data, df], axis=0) #append 
        else:
            print(f"Stock: {stock} is NOT added to final dataset!\n")
            num_failure+=1
            
    if num_failures == len(stock_list):
        raise ValueError("Every data frame extracted is empty. No data is fetched.")
        
    
    return data

In [5]:
stock_list = dow_30_stocks
raw_data = extract_yf_rawdata(start_date, end_date, stock_list)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

In [6]:
print("Columns immediately after extraction:", raw_data.columns.tolist())
print("Sample data:")
print(raw_data.head())
print(f"\n{raw_data.index}\n")

print(f"Collected data: {'-'*30}\n{raw_data}\n{'-'*60}\n")
print(f"Breif Statistics Description: {'-'*30}\n{raw_data.describe()}\n{'-'*60}")

print(f"\nNote: Original Method based on old ticker list produced (94301,8)\n")

Columns immediately after extraction: ['Close', 'High', 'Low', 'Open', 'Volume', 'Stock']
Sample data:
                Close       High        Low       Open   Volume Stock
Date                                                                 
2000-01-03  19.516684  19.956132  19.452060  19.865658  2599386   MMM
2000-01-04  18.741198  19.607169  18.741198  19.206494  3245705   MMM
2000-01-05  19.284048  19.904445  18.844599  18.844599  4424482   MMM
2000-01-06  20.835035  21.196934  19.503765  19.503765  7147057   MMM
2000-01-07  21.248638  21.468364  20.667016  20.912589  4905035   MMM

DatetimeIndex(['2000-01-03', '2000-01-04', '2000-01-05', '2000-01-06',
               '2000-01-07', '2000-01-10', '2000-01-11', '2000-01-12',
               '2000-01-13', '2000-01-14',
               ...
               '2025-07-18', '2025-07-21', '2025-07-22', '2025-07-23',
               '2025-07-24', '2025-07-25', '2025-07-28', '2025-07-29',
               '2025-07-30', '2025-07-31'],
              dt

Raw data is extracted from yfinance API. 
### Step 2: Data Formatting
Perform elementary data transformation to create a data frame compliant for formatting.

####  Clean and Reformat raw data

In [7]:
def format_raw_data(data:pd.DataFrame)->pd.DataFrame:
    """
    Returns a formatted data frame with following formatting:
    1. drops missing data by rows,
    2. resets index as integer indices 'Index' instead of ordered by date,
    3. reorders the columns by Date and then Stock Name is ascending format,
    4. reformats index,
    5. creates an additional 'Day' column to include 'day effects' in financial market,
    6. drops old index completely.

    Integration based on: https://github.com/AI4Finance-Foundation/FinRL/blob/master/finrl/meta/preprocessor/yahoodownloader.py#L36
    """    
    data = data.reset_index() 
    data.index.name='Index'

    #Adding day-of-week column to address the "day effects" in financial market
    data['Day'] = data['Date'].dt.dayofweek #.dt (optimized C-level datetime data), dayofweek
    
    # data['Date'] = data.Date.apply(lambda x: x.strftime("%Y-%M-%D"))
    data=data.dropna().reset_index(drop=True)
    data = data.sort_values(by=['Date', 'Stock']).reset_index(drop=True)

    # print(data.head(31))

    return data

In [8]:
data = format_raw_data(raw_data)
# print(data.head(20))
# print(data.)

### Step3: Preprocess Data: 

#### Feature Engineering Decisions and Process:
1. ##### Adding industry-standard **technical indicators** to indicate comprehensive market information.

Because indicators are mathematical caculations derived from price, volume, these indicators will be engineered as additional features to help the DRL agents better identify market trends, momentum, and potential reversal points. The selected indicators balance different time horizons, multiple signal types, and includes complementary indicators that work well together. This combination list aligns with industry best practices and provides efficient data samples for DRL learning.

Criteria of selecting technical indicators is based on common pratice on trading platform and quantitative library. Selected indicators are the most widely implemented ones representing over 80% usage on trading platform, thus the selection ensures realistic learning for agents.

Selected indicators: 

**Momentum & Trend: macd** Moving Average Convergence Divergence, the gold standard for trend identification and momentum shifts;

**Volatility: boll_ub, boll_lb** Bollinger Bands Upper (standard 20-period), Bollinger Bands Lower (standard 20-period);

**Trend Following: close_30_sma, close_60_sma** 30-period Simple Moving Average, 60-period Simple Moving Average;

**Oscillators: rsi_14, rsi_30** 14-period Relative Strength Index (standard RSI), 30-period Relative Strength Index, for overbought/oversold conditions;

**Oscillators: cci_30** 30-period Commodity Channel Index, for cyclical trend detection;

**Oscillators: dx_30** 30-period Directional Movement Index, for trend strength measurement;

NOTE:
The list is highly inspired by FinRL library's INDICATOR list selection, and includes additional **rsi_14** to for optimized industry alignment.


2. ##### Adding **turbulence index**, to reflect market volatility level and control risk in worst-case scenario. The index responsibily measures fluctuation of asset prices and trains agents to make informed decisions.




In [9]:
print(data.head(20))
tech_indicators = ['macd', 'boll_ub', 'boll_lb', 'rsi_14', 'rsi_30', 'cci_30', 'dx_30', 'close_30_sma', 'close_60_sma']

         Date      Close       High        Low       Open     Volume Stock  \
0  2000-01-03   0.840094   0.844316   0.763168   0.787090  535796800  AAPL   
1  2000-01-03  43.521145  48.404848  43.477926  48.404848   22914900  AMGN   
2  2000-01-03   4.468750   4.478125   3.952344   4.075000  322352000  AMZN   
3  2000-01-03  32.405487  33.899541  32.147892  33.899541    6471267   AXP   
4  2000-01-03  25.940283  26.908505  25.698227  26.747135    2638200    BA   
5  2000-01-03  12.610173  12.707424  12.367046  12.367046    5055000   CAT   
6  2000-01-03  35.360302  36.076098  33.887808  35.973841   53076000  CSCO   
7  2000-01-03  16.231289  16.668005  16.025062  16.668005    4387600   CVX   
8  2000-01-03  22.736221  22.783787  21.880046  22.260569    8402230   DIS   
9  2000-01-03  61.993744  66.249507  61.598879  66.117885    1822600    GS   
10 2000-01-03  38.380390  40.735467  37.570833  40.404284   12030800    HD   
11 2000-01-03  30.489410  31.464264  30.388563  31.027261    220

In [10]:
def transform_data(data):
    """
    1. If a stock is not present in one day, the entire stock is eliminated from the dataset.
    2. Converts dates to sequential integers for models.
        This ensures memory efficiency and correct numerical model input.
    3. Transforms each stock into a column/attribute for model to learn, based on 'Close' values, indexed by 'Date'.
    4. 
    """
    df = data.copy()

    #assigns each unique date an integer index
    df.index, _ =df.Date.factorize()
    
    stocks_df = df.pivot_table(index='Date', columns='Stock', values='Close')

    # clean all columns with null values (removes the stock)
    stocks_df = stocks_df.dropna(axis=1) 

    # merge the transformed stocks df into original df
    stocks = stocks_df.columns
    df=df[df.Stock.isin(stocks)]

    print(df)

    return df

In [11]:
data = transform_data(data)

           Date       Close        High         Low        Open     Volume  \
0    2000-01-03    0.840094    0.844316    0.763168    0.787090  535796800   
0    2000-01-03   43.521145   48.404848   43.477926   48.404848   22914900   
0    2000-01-03    4.468750    4.478125    3.952344    4.075000  322352000   
0    2000-01-03   32.405487   33.899541   32.147892   33.899541    6471267   
0    2000-01-03   25.940283   26.908505   25.698227   26.747135    2638200   
...         ...         ...         ...         ...         ...        ...   
6432 2025-07-31  330.880005  334.040009  329.480011  329.480011    1752600   
6432 2025-07-31  260.239990  263.390015  257.929993  257.929993    1744800   
6432 2025-07-31  249.559998  261.399994  247.750000  261.399994   29606900   
6432 2025-07-31   42.759998   43.110001   42.150002   42.290001   25538700   
6432 2025-07-31   97.980003   98.629997   97.309998   97.440002   15651300   

     Stock  Day  
0     AAPL    0  
0     AMGN    0  
0     AMZ

In [12]:
from stockstats import StockDataFrame as Sdf
def engineer_indicators(indicators, data):
    """
    Calculates technical indicators and engineers the values into the dataframe.
    """ 
    df = data.copy()
    stock_sdf = Sdf.retype(df.copy())
    stocks_list = stock_sdf.stock.unique()

    
    for indicator in indicators:
        "Every indicator will be appended .."
        indicator_df = pd.DataFrame() #a new df every indicator
        size = len(stocks_list)

        for i in range(size):
            try:
                # print(f"0. stock sdf parsed: \n{stock_sdf}")
                relevant_stock_df = stock_sdf[stock_sdf.stock == stocks_list[i]]
                # print(f"1. Relevant stock df: \n{relevant_stock_df.head(20)}\n")
                
                calculations = relevant_stock_df[indicator]
                # print(f"2. The indicator referenced: {indicator}: with the values\n{calculations.head(10)}")
                calculations = pd.DataFrame(calculations)
        
                calculations['Stock']=stocks_list[i]
                # print(f"3. calculations: \n{calculations}")
                calculations['Date'] = df[df.Stock == stocks_list[i]]['Date'].to_list()
                # print(f"4. calculations: \n{calculations}")
                
                indicator_df=pd.concat(
                    [indicator_df, calculations], axis=0, ignore_index=True
                )
                # print(f"5. indicator df for indicator {indicator} for stock {stocks_list[i]}: \n{indicator_df}")
                
            except Exception as e:
                print("Hey!", e)
        # print(f"6. indicator list after all stocks calculations: \n{indicator_df}")   
        
        df=df.merge(
            indicator_df[[indicator, 'Stock', 'Date']], on=['Stock','Date'],how='left'
        )
        
        # print(f"7. final merged df with {indicator} for all stock values computed included: \n{df}")
        
    df=df.sort_values(by=['Date','Stock'])
    print(f"8. final df:\n{df}")
    
    return df

In [13]:
data = engineer_indicators(tech_indicators, data)

8. final df:
             Date       Close        High         Low        Open     Volume  \
0      2000-01-03    0.840094    0.844316    0.763168    0.787090  535796800   
1      2000-01-03   43.521145   48.404848   43.477926   48.404848   22914900   
2      2000-01-03    4.468750    4.478125    3.952344    4.075000  322352000   
3      2000-01-03   32.405487   33.899541   32.147892   33.899541    6471267   
4      2000-01-03   25.940283   26.908505   25.698227   26.747135    2638200   
...           ...         ...         ...         ...         ...        ...   
180119 2025-07-31  330.880005  334.040009  329.480011  329.480011    1752600   
180120 2025-07-31  260.239990  263.390015  257.929993  257.929993    1744800   
180121 2025-07-31  249.559998  261.399994  247.750000  261.399994   29606900   
180122 2025-07-31   42.759998   43.110001   42.150002   42.290001   25538700   
180123 2025-07-31   97.980003   98.629997   97.309998   97.440002   15651300   

       Stock  Day       ma

In [14]:
def add_vix(data, start_date, end_date):
    df = data.copy()
    # print(df)
    vix = yf.download(
        "^VIX",
        start=start_date,
        end=end_date,
        auto_adjust=True
    )
    if vix.columns.nlevels != 1:
        vix.columns = vix.columns.droplevel(1)
        
    vix = vix.reset_index()
    # print(vix)
    vix_data = vix[["Date", "Close"]]
    vix_data.columns=["Date","VIX_Close"]
                 
    df = df.merge(vix_data, on="Date")
    df = df.sort_values(["Date", "Stock"]).reset_index(drop=True)
    # print(df.head(30))
    print("Vix added successfully!")

    return df

In [15]:
data = add_vix(data, start_date, end_date)

[*********************100%***********************]  1 of 1 completed

Vix added successfully!


#### Engineer Turbulence

In [20]:
def engineer_turbulence(data):
    """
    Turbulence measures regime shifts requiring tactical allocation changes.
    It complements VIX and measures actual breakdown of market structure.
    """
    df = data.copy()
    # calculate turbulence
    turbulence_index = []
    df_price_pivot = df.pivot(index='Date', columns='Stock', values='Close')
    df_price_pivot = df_price_pivot.pct_change()

    unique_date=df.Date.unique()
    #start after a year
    start=252
    turbulence_index=[0]*start
    count=0
    for i in range(start, len(unique_date)):
        current_price = df_price_pivot[df_price_pivot.index==unique_date[i]]
        #one year rolling window to calculate covariance
        hist_price=df_price_pivot[
            (df_price_pivot.index < unique_date[i]) &
            (df_price_pivot.index >= unique_date[i-252])
        ]
        #drop tickers which has number missing values more than the oldest ticker
        filtered_hist_price=hist_price.iloc[hist_price.isna().sum().min():].dropna(axis=1)
        cov_temp=filtered_hist_price.cov()
        current_temp=current_price[[x for x in filtered_hist_price]] - np.mean(filtered_hist_price, axis=0)
        temp=current_temp.values.dot(np.linalg.pinv(cov_temp)).dot(current_temp.values.T)
        if temp>0:
            count+=1
            if count > 2:
                turbulence_temp=temp[0][0]
            else:
                # avoid large outlier, because the calculations just begins
                turbulence_temp=0
        else:
            turbulence_temp=0
        turbulence_index.append(turbulence_temp)

    try:
        turbulence_index=pd.DataFrame(
            {'Date':df_price_pivot.index, 'Turbulence': turbulence_index}
        )
    except ValueError:
        raise Exception("Turbulence information could not be added.")

    #merge df
    df = df.merge(turbulence_index, on='Date')
    df=df.sort_values(['Date','Stock']).reset_index(drop=True)
    return df
    

In [22]:
data = engineer_turbulence(data)

KeyboardInterrupt: 

In [23]:
print(data)

             Date       Close        High         Low        Open     Volume  \
0      2000-01-03    0.840094    0.844316    0.763168    0.787090  535796800   
1      2000-01-03   43.521145   48.404848   43.477926   48.404848   22914900   
2      2000-01-03    4.468750    4.478125    3.952344    4.075000  322352000   
3      2000-01-03   32.405487   33.899541   32.147892   33.899541    6471267   
4      2000-01-03   25.940283   26.908505   25.698227   26.747135    2638200   
...           ...         ...         ...         ...         ...        ...   
180119 2025-07-31  330.880005  334.040009  329.480011  329.480011    1752600   
180120 2025-07-31  260.239990  263.390015  257.929993  257.929993    1744800   
180121 2025-07-31  249.559998  261.399994  247.750000  261.399994   29606900   
180122 2025-07-31   42.759998   43.110001   42.150002   42.290001   25538700   
180123 2025-07-31   97.980003   98.629997   97.309998   97.440002   15651300   

       Stock  Day       macd     boll_u

### Step4: Store Data:

#### Split data for training and trading
How much to allocate for training and for testing? By 80-20?

In [36]:
def split_data(data, start, end, target_col='Date'):
    "Split the data by start and end date."
    # print(f"0. data: \n{data}")
    result =data[(data[target_col] >= start) & (data[target_col] < end)]
    # print(f"1. result: \n{result}")
    result = result.sort_values([target_col, 'Stock'], ignore_index=True)
    # print(f"2. result: \n{result}")
    result.index=result[target_col].factorize()[0]
    # print(f"3. result: \n{result}")
    return result

In [35]:
split_data(data, '2000-01-02', '2003-01-02')

1. result: 
            Date      Close       High        Low       Open     Volume Stock  \
0     2000-01-03   0.840094   0.844316   0.763168   0.787090  535796800  AAPL   
1     2000-01-03  43.521145  48.404848  43.477926  48.404848   22914900  AMGN   
2     2000-01-03   4.468750   4.478125   3.952344   4.075000  322352000  AMZN   
3     2000-01-03  32.405487  33.899541  32.147892  33.899541    6471267   AXP   
4     2000-01-03  25.940283  26.908505  25.698227  26.747135    2638200    BA   
...          ...        ...        ...        ...        ...        ...   ...   
21051 2002-12-31   6.852503   6.901016   6.716666   6.820969    1244100   SHW   
21052 2002-12-31  19.693001  19.693001  19.403823  19.548412     622400   TRV   
21053 2002-12-31  16.423227  16.509769  16.248177  16.444863    4596800   UNH   
21054 2002-12-31  11.276221  11.418810  11.180191  11.348971    6211452    VZ   
21055 2002-12-31  10.961313  10.985185  10.796383  10.980845   23478900   WMT   

       Day     

,Date,Close,High,Low,Open,Volume,Stock,Day,macd,boll_ub,boll_lb,rsi_14,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,VIX_Close,Turbulence
0,2000-01-03,0.840094,0.844316,0.763168,0.787090,535796800,AAPL,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.840094,0.840094,24.209999,0.000000
0,2000-01-03,43.521145,48.404848,43.477926,48.404848,22914900,AMGN,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,43.521145,43.521145,24.209999,0.000000
0,2000-01-03,4.468750,4.478125,3.952344,4.075000,322352000,AMZN,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,4.468750,4.468750,24.209999,0.000000
0,2000-01-03,32.405487,33.899541,32.147892,33.899541,6471267,AXP,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,32.405487,32.405487,24.209999,0.000000
0,2000-01-03,25.940283,26.908505,25.698227,26.747135,2638200,BA,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,25.940283,25.940283,24.209999,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
751,2002-12-31,6.852503,6.901016,6.716666,6.820969,1244100,SHW,1,0.032520,7.062083,6.649230,51.053617,52.553782,-29.322854,15.005141,6.865359,6.565653,28.620001,16.845632
751,2002-12-31,19.693001,19.693001,19.403823,19.548412,622400,TRV,1,-0.058451,20.634912,19.121430,49.326174,51.414522,-81.390668,2.368685,20.076999,19.324655,28.620001,16.845632
751,2002-12-31,16.423227,16.509769,16.248177,16.444863,4596800,UNH,1,-0.147421,16.724988,15.744812,49.132498,47.059578,9.037345,24.347433,16.315836,17.404196,28.620001,16.845632
751,2002-12-31,11.276221,11.418810,11.180191,11.348971,6211452,VZ,1,0.029129,11.796222,11.040522,47.720679,52.095907,-68.393493,7.602377,11.511055,11.082292,28.620001,16.845632


#### Save Data to local 
Saving the data as Parquet for memory efficiency.
will need to pip install pyarrow when reading from it

In [61]:
from datetime import datetime
def divide_date_8020(start, end):
    start = pd.to_datetime(start)
    end=pd.to_datetime(end)
    total_days = (end - start).days
    train_days = int(total_days * 0.8)
    
    split= start + pd.Timedelta(days=train_days)
    
    train_start = start
    train_end = split
    trade_start = split + pd.Timedelta(days=1)
    trade_end=end

    print(f"Total period: {start} to {end} ({total_days} days)")
    print(f"Training: {start} to {split.strftime('%Y-%m-%d')}")
    print(f"Testing: {trade_start.strftime('%Y-%m-%d')} to {end}")

    return train_start, train_end, trade_start, trade_end
    

In [62]:
# divide by 80-20
TRAIN_START_DATE, TRAIN_END_DATE, TRADE_START_DATE, TRADE_END_DATE = divide_date_8020(start_date, end_date)

train = split_data(data, TRAIN_START_DATE, TRAIN_END_DATE)
test = split_data(data, TRADE_START_DATE, TRADE_END_DATE)

print(len(train))
print(len(test))

Total period: 2000-01-01 00:00:00 to 2025-08-01 00:00:00 (9344 days)
Training: 2000-01-01 00:00:00 to 2020-06-19
Testing: 2020-06-20 to 2025-08-01 00:00:00
144144
35952


In [59]:
train.to_parquet(train_file_path, engine='pyarrow')
test.to_parquet(test_file_path, engine='pyarrow')

In [60]:
!ls

'Data Extracting and Engineering.ipynb'   test.parquet
 data_representation.ipynb		  train.parquet
